# Fuzzy Mamdani for Fish Feeder Paper

Notebook ini disusun untuk kebutuhan paper dan validasi metodologi sebelum implementasi ke Arduino UNO dan ESP32. Fokus notebook:

- definisi variabel fuzzy
- penentuan membership function
- rule base / rule table
- visualisasi membership function
- proses fuzzifikasi
- inferensi Mamdani `max-min`
- agregasi output
- defuzzifikasi dengan metode centroid
- contoh numerik yang bisa langsung dipakai dalam penulisan paper


## 1. Import Library

Notebook ini sengaja hanya memakai `numpy`, `pandas`, dan `matplotlib` agar mudah direproduksi.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

## 2. Semesta Pembicaraan dan Fungsi Keanggotaan

Pada contoh ini digunakan dua input dan satu output:

- Input 1: suhu air `temp` dalam `degC`
- Input 2: biomassa `biomass` dalam gram
- Output: jumlah pakan `feed_output` dalam gram

Bentuk membership function dibuat sederhana dan mudah dipindahkan ke embedded system: kombinasi segitiga dan trapesium.

In [ ]:
temp_universe = np.linspace(20, 40, 2001)
biomass_universe = np.linspace(0, 2000, 2001)
feed_universe = np.linspace(0, 60, 2001)


def trimf(x, a, b, c):
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    if b != a:
        left = (x >= a) & (x <= b)
        y[left] = (x[left] - a) / (b - a)
    if c != b:
        right = (x >= b) & (x <= c)
        y[right] = np.maximum(y[right], (c - x[right]) / (c - b))
    y[x == b] = 1.0
    return np.clip(y, 0.0, 1.0)


def trapmf(x, a, b, c, d):
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    if b != a:
        left = (x >= a) & (x <= b)
        y[left] = (x[left] - a) / (b - a)
    plateau = (x >= b) & (x <= c)
    y[plateau] = 1.0
    if d != c:
        right = (x >= c) & (x <= d)
        y[right] = (d - x[right]) / (d - c)
    return np.clip(y, 0.0, 1.0)

## 3. Definisi Variabel Fuzzy

### 3.1. Suhu Air

Kategori suhu air:

- `cold`
- `normal`
- `warm`
- `hot`

Rentang dapat disesuaikan dengan hasil eksperimen aktual.

In [ ]:
temp_mf = {
    'cold': trapmf(temp_universe, 20, 20, 23, 26),
    'normal': trimf(temp_universe, 24, 27.5, 31),
    'warm': trimf(temp_universe, 29, 32.5, 36),
    'hot': trapmf(temp_universe, 34, 36, 40, 40),
}

plt.figure(figsize=(9, 5))
for label, values in temp_mf.items():
    plt.plot(temp_universe, values, label=label)
plt.title('Membership Function for Water Temperature')
plt.xlabel('Temperature (degC)')
plt.ylabel('Membership Degree')
plt.ylim(-0.02, 1.05)
plt.legend()
plt.show()

### 3.2. Biomassa Ikan

Kategori biomassa:

- `low`
- `medium`
- `high`

In [ ]:
biomass_mf = {
    'low': trapmf(biomass_universe, 0, 0, 300, 700),
    'medium': trimf(biomass_universe, 500, 1000, 1500),
    'high': trapmf(biomass_universe, 1200, 1600, 2000, 2000),
}

plt.figure(figsize=(9, 5))
for label, values in biomass_mf.items():
    plt.plot(biomass_universe, values, label=label)
plt.title('Membership Function for Fish Biomass')
plt.xlabel('Biomass (g)')
plt.ylabel('Membership Degree')
plt.ylim(-0.02, 1.05)
plt.legend()
plt.show()

### 3.3. Output Pakan

Kategori output pakan:

- `very_low`
- `low`
- `medium`
- `high`
- `very_high`

In [ ]:
feed_mf = {
    'very_low': trapmf(feed_universe, 0, 0, 5, 12),
    'low': trimf(feed_universe, 8, 16, 24),
    'medium': trimf(feed_universe, 20, 30, 40),
    'high': trimf(feed_universe, 36, 45, 54),
    'very_high': trapmf(feed_universe, 48, 55, 60, 60),
}

plt.figure(figsize=(9, 5))
for label, values in feed_mf.items():
    plt.plot(feed_universe, values, label=label)
plt.title('Membership Function for Feed Output')
plt.xlabel('Feed Output (g)')
plt.ylabel('Membership Degree')
plt.ylim(-0.02, 1.05)
plt.legend()
plt.show()

## 4. Rule Base Mamdani

Rule base berikut adalah contoh akademik yang logis untuk feeder:

- semakin tinggi biomassa, umumnya pakan meningkat
- suhu rendah cenderung menurunkan pakan
- suhu optimal atau hangat cenderung meningkatkan pakan
- suhu sangat tinggi dapat ditahan agar tidak berlebihan

Format rule:

`IF temperature IS ... AND biomass IS ... THEN feed IS ...`

In [ ]:
rules = [
    ('cold', 'low', 'very_low'),
    ('cold', 'medium', 'low'),
    ('cold', 'high', 'medium'),
    ('normal', 'low', 'low'),
    ('normal', 'medium', 'medium'),
    ('normal', 'high', 'high'),
    ('warm', 'low', 'medium'),
    ('warm', 'medium', 'high'),
    ('warm', 'high', 'very_high'),
    ('hot', 'low', 'low'),
    ('hot', 'medium', 'medium'),
    ('hot', 'high', 'high'),
]

rule_df = pd.DataFrame(rules, columns=['temperature', 'biomass', 'feed_output'])
rule_df

In [ ]:
rule_table = rule_df.pivot(index='temperature', columns='biomass', values='feed_output')
rule_table

## 5. Fungsi Bantu Fuzzifikasi dan Inferensi

Inferensi Mamdani yang dipakai:

- operator AND: `min`
- implikasi: `min`
- agregasi antar rule: `max`
- defuzzifikasi: centroid

In [ ]:
def interp_membership(universe, mf_values, crisp_value):
    return float(np.interp(crisp_value, universe, mf_values))


def fuzzify_inputs(temp_value, biomass_value):
    temp_deg = {
        label: interp_membership(temp_universe, mf, temp_value)
        for label, mf in temp_mf.items()
    }
    biomass_deg = {
        label: interp_membership(biomass_universe, mf, biomass_value)
        for label, mf in biomass_mf.items()
    }
    return temp_deg, biomass_deg


def mamdani_inference(temp_value, biomass_value):
    temp_deg, biomass_deg = fuzzify_inputs(temp_value, biomass_value)
    clipped_outputs = []
    firing_rows = []

    for idx, (temp_label, biomass_label, output_label) in enumerate(rules, start=1):
        alpha = min(temp_deg[temp_label], biomass_deg[biomass_label])
        clipped = np.minimum(alpha, feed_mf[output_label])
        clipped_outputs.append(clipped)
        firing_rows.append({
            'rule': idx,
            'if_temp': temp_label,
            'if_biomass': biomass_label,
            'then_feed': output_label,
            'mu_temp': temp_deg[temp_label],
            'mu_biomass': biomass_deg[biomass_label],
            'alpha': alpha,
        })

    aggregated = np.max(np.vstack(clipped_outputs), axis=0)
    denominator = np.sum(aggregated)
    centroid = np.sum(feed_universe * aggregated) / denominator if denominator > 0 else 0.0

    return {
        'temp_deg': temp_deg,
        'biomass_deg': biomass_deg,
        'firing_df': pd.DataFrame(firing_rows),
        'aggregated': aggregated,
        'centroid': float(centroid),
    }

## 6. Contoh Kasus Fuzzifikasi

Pilih satu contoh crisp input. Nilai ini dapat diganti sesuai skenario pada paper.

In [ ]:
temp_sample = 30.0
biomass_sample = 1200.0

result = mamdani_inference(temp_sample, biomass_sample)
result['centroid']

In [ ]:
pd.DataFrame({
    'temperature_label': list(result['temp_deg'].keys()),
    'membership_degree': list(result['temp_deg'].values()),
})

In [ ]:
pd.DataFrame({
    'biomass_label': list(result['biomass_deg'].keys()),
    'membership_degree': list(result['biomass_deg'].values()),
})

## 7. Visualisasi Fuzzifikasi Input

Garis vertikal menunjukkan nilai crisp input.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for label, mf in temp_mf.items():
    axes[0].plot(temp_universe, mf, label=label)
axes[0].axvline(temp_sample, color='black', linestyle='--', label=f'temp={temp_sample}')
axes[0].set_title('Fuzzification of Temperature Input')
axes[0].set_xlabel('Temperature (degC)')
axes[0].set_ylabel('Membership Degree')
axes[0].legend()

for label, mf in biomass_mf.items():
    axes[1].plot(biomass_universe, mf, label=label)
axes[1].axvline(biomass_sample, color='black', linestyle='--', label=f'biomass={biomass_sample}')
axes[1].set_title('Fuzzification of Biomass Input')
axes[1].set_xlabel('Biomass (g)')
axes[1].set_ylabel('Membership Degree')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Aktivasi Rule dan Nilai Alpha-Predikat

Tabel berikut memperlihatkan firing strength setiap rule.

In [ ]:
firing_df = result['firing_df'].copy()
firing_df.sort_values(by='alpha', ascending=False).reset_index(drop=True)

## 9. Visualisasi Output Rule yang Aktif

Setiap rule memotong membership function output sesuai nilai `alpha`. Grafik ini berguna untuk menjelaskan inferensi Mamdani pada paper.

In [ ]:
active_rules = firing_df[firing_df['alpha'] > 0].copy()
active_rules

In [ ]:
plt.figure(figsize=(10, 5))

for _, row in active_rules.iterrows():
    clipped = np.minimum(row['alpha'], feed_mf[row['then_feed']])
    plt.plot(feed_universe, clipped, label=f"R{int(row['rule'])}: {row['then_feed']} (alpha={row['alpha']:.2f})")

plt.title('Clipped Output of Active Mamdani Rules')
plt.xlabel('Feed Output (g)')
plt.ylabel('Membership Degree')
plt.ylim(-0.02, 1.05)
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.show()

## 10. Agregasi Output dan Defuzzifikasi

Agregasi dilakukan dengan operator `max`. Nilai crisp output diperoleh menggunakan metode centroid.

In [ ]:
aggregated = result['aggregated']
centroid = result['centroid']

plt.figure(figsize=(10, 5))
for label, mf in feed_mf.items():
    plt.plot(feed_universe, mf, linestyle='--', alpha=0.35, label=f'{label} base')

plt.fill_between(feed_universe, aggregated, color='tab:blue', alpha=0.4, label='Aggregated output')
plt.axvline(centroid, color='red', linestyle='-', linewidth=2, label=f'Centroid = {centroid:.2f} g')
plt.title('Aggregated Mamdani Output and Centroid Defuzzification')
plt.xlabel('Feed Output (g)')
plt.ylabel('Membership Degree')
plt.ylim(-0.02, 1.05)
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.show()

## 11. Ringkasan Hasil Contoh Kasus

Tabel ini dapat langsung dipakai sebagai ringkasan hasil inferensi pada paper.

In [ ]:
summary_df = pd.DataFrame([
    {
        'temp_input_degC': temp_sample,
        'biomass_input_g': biomass_sample,
        'defuzzified_feed_output_g': centroid,
        'active_rule_count': int((firing_df['alpha'] > 0).sum()),
    }
])
summary_df

## 12. Simulasi Grid untuk Analisis Permukaan

Bagian ini berguna untuk menunjukkan bagaimana output fuzzy berubah terhadap kombinasi suhu dan biomassa. Grafik ini sering dipakai di paper untuk memperlihatkan karakter sistem fuzzy.

In [ ]:
temp_grid = np.arange(24, 37, 1)
biomass_grid = np.arange(200, 2001, 200)

surface_rows = []
for t in temp_grid:
    for b in biomass_grid:
        out = mamdani_inference(float(t), float(b))['centroid']
        surface_rows.append({'temp': t, 'biomass': b, 'feed_output': out})

surface_df = pd.DataFrame(surface_rows)
surface_df.head()

In [ ]:
pivot_surface = surface_df.pivot(index='biomass', columns='temp', values='feed_output')

plt.figure(figsize=(10, 6))
plt.imshow(
    pivot_surface.values,
    aspect='auto',
    origin='lower',
    extent=[pivot_surface.columns.min(), pivot_surface.columns.max(), pivot_surface.index.min(), pivot_surface.index.max()],
    cmap='viridis'
)
plt.colorbar(label='Defuzzified Feed Output (g)')
plt.title('Response Surface of Mamdani Fuzzy System')
plt.xlabel('Temperature (degC)')
plt.ylabel('Biomass (g)')
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

T, B = np.meshgrid(pivot_surface.columns.values, pivot_surface.index.values)
Z = pivot_surface.values

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(T, B, Z, cmap='viridis', edgecolor='k', linewidth=0.2, alpha=0.9)
ax.set_title('3D Surface of Fuzzy Mamdani Output')
ax.set_xlabel('Temperature (degC)')
ax.set_ylabel('Biomass (g)')
ax.set_zlabel('Feed Output (g)')
plt.show()

## 13. Template Narasi untuk Paper

Narasi yang bisa diadaptasi:

1. Sistem inferensi fuzzy Mamdani menggunakan dua variabel input, yaitu suhu air dan biomassa ikan, serta satu variabel output berupa jumlah pakan.
2. Masing-masing variabel direpresentasikan menggunakan fungsi keanggotaan segitiga dan trapesium agar mudah dihitung dan diimplementasikan pada perangkat embedded.
3. Basis aturan dibentuk dari relasi logis antara suhu, biomassa, dan jumlah pakan, dengan operator AND menggunakan minimum.
4. Agregasi output dilakukan dengan operator maksimum, sedangkan defuzzifikasi dilakukan menggunakan metode centroid.
5. Hasil simulasi menunjukkan bahwa kenaikan biomassa cenderung meningkatkan output pakan, sementara perubahan suhu memodulasi jumlah pakan sesuai zona termal yang didefinisikan.

Silakan sesuaikan parameter membership function dan rule base agar identik dengan model final yang akan dipakai di firmware.